# Missing Flags Study

In [11]:
from pathlib import Path
import pickle
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
from temporal_metrics import temp_metrics

In [13]:
dataset = "aitv2"
scenario = "santos"
test_scenario = "santos"

sim_start = pd.Timestamp("2022-01-14 18:00")
attack_start = pd.Timestamp("2022-01-17 11:00")
attack_end   = pd.Timestamp("2022-01-17 12:00")

In [14]:
phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
}

In [15]:
experiment = "miss_flags_study"
experiment_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}/")
reports_dir = Path(f"../../reports/{dataset}/{scenario}/{experiment}")

In [16]:
def compute_weighted_f1(metrics):
    total_support = 0
    weighted_f1_sum = 0

    for class_name, stats in metrics['per_class'].items():
        support = stats['TP'] + stats['FN']
        total_support += support
        weighted_f1_sum += stats['f1'] * support

    weighted_f1 = weighted_f1_sum / total_support

    return weighted_f1

In [17]:
experiments = {}

In [18]:
metrics_dir = experiment_dir / "metrics"
file_paths = list(metrics_dir.iterdir())

for file_path in file_paths:
    data = np.load(file_path, allow_pickle=True)
    metrics = data["metrics"].item()

    experiment_name = str(file_path.stem)
    print(f"Experiment: {experiment_name}")
    
    weighted_f1 = compute_weighted_f1(metrics)
    
    experiments[experiment_name] = {
        "f1_score": metrics["macro_f1"],
        "weighted_f1": weighted_f1,
        "false_alarm_rate": metrics["false_alarm_rate"],
        "detection_rate": metrics["detection_rate"]
    }
    print(weighted_f1)
    for phase in metrics["per_class"]:
        print(f"Phase: {phase}")
        print(f"  F1 Score: {metrics['per_class'][phase]}")
    print()
    # print(metrics["per_class"]["benign"])

Experiment: ait_temp_context_endtoend_base_w100_full_0.001lr_20260521_235742
0.9989589334741009
Phase: benign
  F1 Score: {'TP': 100107, 'FP': 54, 'FN': 56, 'precision': 0.999460868002516, 'recall': 0.9994409113145573, 'f1': 0.9994508895589146}
Phase: phase1
  F1 Score: {'TP': 3810, 'FP': 47, 'FN': 46, 'precision': 0.9878143634949442, 'recall': 0.9880705394190872, 'f1': 0.9879424348502529}
Phase: phase2
  F1 Score: {'TP': 1701, 'FP': 0, 'FN': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Phase: phase3
  F1 Score: {'TP': 5, 'FP': 0, 'FN': 3, 'precision': 1.0, 'recall': 0.625, 'f1': 0.7692307692307693}
Phase: phase4
  F1 Score: {'TP': 63, 'FP': 9, 'FN': 5, 'precision': 0.875, 'recall': 0.9264705882352942, 'f1': 0.9}

Experiment: ait_miss_flags_endtoend_base_w100_full_0.001lr_20260521_163337
0.998085550785013
Phase: benign
  F1 Score: {'TP': 100072, 'FP': 106, 'FN': 91, 'precision': 0.9989418834474635, 'recall': 0.9990914808861555, 'f1': 0.9990166765664541}
Phase: phase1
  F1 Score: {'TP